In [7]:

import ipywidgets as widgets
from ipyleaflet import Map, Polyline, LayerGroup, LayersControl
import numpy as np
import pandas as pd
import o2a

urns = ['vessel:mya_ii:moses_moblab:pfb_awi_751801:latitude_0001', 
    'vessel:mya_ii:moses_moblab:pfb_awi_751801:longitude_0001',
    'vessel:littorina:moses_moblab:pfb_awi_751801:latitude_0001',
    'vessel:littorina:moses_moblab:pfb_awi_751801:longitude_0001', 
    'vessel:prandtl_hzg:moses_moblab:pfb_awi_751801:latitude_0001', 
    'vessel:prandtl_hzg:moses_moblab:pfb_awi_751801:longitude_0001', 
    'drifter:moses-drifter-304:latitude',
    'drifter:moses-drifter-304:longitude',
    'drifter:moses-drifter-306:latitude',
    'drifter:moses-drifter-306:longitude',
    'drifter:moses-drifter-319:latitude',
    'drifter:moses-drifter-319:longitude',
    'drifter:moses-drifter-309:latitude',
    'drifter:moses-drifter-309:longitude',
    'drifter:moses-drifter-312:latitude',
    'drifter:moses-drifter-312:longitude',
    'drifter:moses-drifter-323:latitude',
    'drifter:moses-drifter-323:longitude',
    'drifter:moses-drifter-324:latitude',
    'drifter:moses-drifter-324:longitude',
    'drifter:moses-drifter-325:latitude',
    'drifter:moses-drifter-325:longitude',
    ]

# Define the date ranges and URN indices for each dataset
datasets = [
    ('2023-09-10T00:00:00', '2023-09-15T00:00:00', urns[0:2]), # mya
    ('2023-09-03T00:00:00', '2023-09-08T00:00:00', urns[2:4]), # littorina
    ('2023-08-31T00:00:00', '2023-09-30T00:00:00', urns[4:]) # rest with same date range
]
a = o2a.o2a()

aggregate='Minute'
aggregateFunctions='MEAN'

# Download data for each dataset and concatenate them
positions_list = [
    a.downloadDataFromDWS(items=",".join(urn_set), begin=begin_date, end=end_date, aggregate=aggregate)
    for begin_date, end_date, urn_set in datasets
]

# Concatenate the data into a single DataFrame
positions = pd.concat(positions_list, ignore_index=True)
names = [urn.split(':')[1] for urn in urns]
names = list(dict.fromkeys(names)) # remove duplicates

# create a method that returns a route for each vessel, given the column indexes of the latitude and longitude of that vessel
def createRoute(positions, latIndex, lonIndex):
    route = []  
    for i in range(len(positions)):
        if not (np.isnan(positions.iloc[i, latIndex]) or np.isnan(positions.iloc[i, lonIndex])):
            route.append((positions.iloc[i, latIndex], positions.iloc[i, lonIndex]))
    return route

# create a list of routes for each vessel in names based on the positions dataframe given the column indexes of the latitude and longitude of that vessel, the first vessel hast column 1 and 2, the second vessel has column 3 and 4, etc.
routes = [createRoute(positions, i*2+1, i*2+2) for i in range(len(names))]

# Create the map with the custom WMS basemap
m = Map(center=(54.2, 8.3), zoom=10, layout=widgets.Layout(width='1400px', height='800px'))

# Define a color table
color_table = [
    "#8B0000",  # Dark Red
    "#556B2F",  # Dark Olive Green
    "#4682B4",  # Steel Blue
    "#D2691E",  # Chocolate
    "#8A2BE2",  # Blue Violet
    "#708090",  # Slate Grey
    "#BDB76B",  # Dark Khaki
    "#C71585",  # Medium Violet Red
    "#A0522D",  # Sienna
    "#2F4F4F",  # Dark Slate Grey
    "#5F9EA0"   # Cadet Blue
]

# Define the routes with names
routes = [
    (1, 2, "Mya II"),
    (3, 4, "Littorina"),
    (5, 6, "drifter_304"),
    (7, 8, "Prandtl"),
    (9, 10, "drifter_306"),
    (11, 12, "drifter_319"),
    (13, 14, "drifter_309"),
    (15, 16, "drifter_312"),
    (17, 18, "drifter_323"),
    (19, 20, "drifter_324"),
    (21, 22, "drifter_325")
]


# Loop through the routes to create polylines and layer groups
for idx, (start_idx, end_idx, name) in enumerate(routes):
    route = createRoute(positions, start_idx, end_idx)
    color = color_table[idx % len(color_table)]  # Get color by index
    line = Polyline(locations=route, color=color, weight=1, opacity=0.8)
    label = f'<span style="color:{color};">{name}</span>'
    layer = LayerGroup(layers=(line,), name=label)
    m.add_layer(layer)

# Add LayerControl
control = LayersControl(position='topright', collapsed=False)
m.add_control(control)

# Display the map (works in Jupyter Notebook)
m

Map(center=[54.2, 8.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…